In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# Example Notebook
This notebook demonstrates how to prepare data, give that data to the sparse bias model, and analyze the results. 

#### Linking Stan
The model is built around the stan code, which is a PPL that leverages Hamiltonian and Markov Chain Monte Carlo to draw samples from complex distributions.
The first step is to link the stan command line tool `cmdstan` to your python environment.
More instructions on this can be found at https://mc-stan.org/cmdstanpy/.
This may have been done automatically if cmdstanpy was installed using conda.

In [ ]:
## may be needed to ensure conda compilers are the same as compiler used with cmdstan build
# import os
# os.environ["CC"] = "/usr/bin/clang"
# os.environ["CXX"] = "/usr/bin/clang++"

In [ ]:
#%%
from cmdstanpy import cmdstan_path, set_cmdstan_path

set_cmdstan_path("/Users/nwalton/Software/cmdstan-2.36.0/")
print(cmdstan_path())

### Fake data

This example notebook will demonsrate the use of the bias identification model using fake/simulated data. 
This data curation step will vary from problem to problem. 
We've provided some utilities for the user, as well as some problem specific tools related to the AIACHNE and PARADIGM projects (add references).
However, the goal is to make the sparse bias model problem agnostic, requiring some user effort to prepare there data in the proper format.

In [ ]:
from utils.functions import maxwellian 

def make_measurement(xmin, xmax, N_obs, relative_unc):
    x_obs = np.logspace(np.log10(xmin),np.log10(xmax), N_obs)
    y_obs_mean = maxwellian(5, x_obs)
    y_obs_unc = y_obs_mean*relative_unc
    y_obs = np.random.default_rng(seed).multivariate_normal(mean=y_obs_mean, cov=np.diag(y_obs_unc**2))
    return x_obs, y_obs, y_obs_unc

seed = 1
N_exp = [25,8,35, 20]
x_range = np.log10((1e-3,12))
# x_eval = np.linspace(10**x_range[0],10**x_range[1], 12)
x_eval = np.logspace(*x_range, 20)

x_obs1, y_obs1, y_obs1_unc = make_measurement(2e-3, 11, N_exp[0], 0.2) 
x_obs2, y_obs2, y_obs2_unc = make_measurement(1e-1, 11, N_exp[1], 0.2) 
x_obs3, y_obs3, y_obs3_unc = make_measurement(5e-2, 11, N_exp[2], 0.2) 
x_obs4, y_obs4, y_obs4_unc = make_measurement(2e-3, 11, N_exp[3], 0.15) 

## optional bias on exp 2
y_obs2 += x_obs2*-1e-3+ 0.02*x_obs2
y_obs2[y_obs2<0 ]=0
y_obs2[-1] -= x_obs2[-1]*-1e-3+ 0.015*x_obs2[-1]
y_obs2_unc = y_obs2*0.1

In [ ]:
plt.figure()
plt.errorbar(x_obs1, y_obs1, yerr=y_obs1_unc, fmt='.')
plt.errorbar(x_obs2, y_obs2, yerr=y_obs2_unc, fmt='.')
plt.errorbar(x_obs3, y_obs3, yerr=y_obs3_unc, fmt='.')
plt.errorbar(x_obs4, y_obs4, yerr=y_obs4_unc, fmt='.')

# plt.errorbar(all_df["energy"], np.array(stan_data['y']), yerr=np.array(stan_data['s'])*np.array(stan_data['y']), fmt = '.')

# plt.ylim(1e-3, 1)
plt.xscale('log')
plt.yscale('log')
plt.show()

The basic input data structure that the user should curate is a dictionary with the following fields:

- X             : Experimental x-grid (ndarray)
- expt_value    : Experimental y-values (ndarray)
- rel_unc       : Relative uncertainty on experimental y-values (ndarray)
- expt_label    : Label indicating experiment label for each data point (list)
- bias_label    : Label indicating the bias group to which each data point belongs to (list) 
- scale_flag    : Boolean flag indicating if this data point has an arbitraty normalization, if True, the model will fit one normalization factor to data points sharing an expt_label. The scale_flag must be the same for all data points with the same expt_label.
- corr          : Correlation matrix describing the correlation in experimental y-values.

Each field should be a list or array-like object of the same length (N), ordered to match the indices of the array-like objects in the other fields. 
This, of course, excludes the corr field which should be an NxN matrix describing the correlation of the experimental y-values.

This dictionary can be constructed for the entire data set or for each individual data set.
In the latter case, the general utility function processing.compile_data_dicts can be used to combine them. 
This function performs the following actions:

1. Broadcasts scalar values (such as bias_label or scale_flag) to match the size of the other fields
2. Concatenates the fields of each individual dataset into a single dictionary
3. Checks data and converts to proper types
4. OPTIONAL: Truncates data based on a min and max value for X
5. OPTIONAL: Rescales the data (if scale_flag==1) to some nominal provided value. The prior on the scaling is a tight distribution around 1.0, so if the datasets are on wildly different scales then they should be rescaled/normalized to a common value first.

As a part of operation 2, it will also properly handle the correlation matrices, though if you have correlations between data set dicts these will need to be added afterwards.

In [ ]:
data1 = {'X'        : x_obs1,
       'expt_value' : y_obs1,
       'rel_unc'    : y_obs1_unc/y_obs1,
       'expt_label' : 'exp1',
       'bias_label' : 'a',
       'scale_flag' : False,
       'corr'       : np.eye(len(x_obs1)) }

data2 = {'X'        : x_obs2,
       'expt_value' : y_obs2,
       'rel_unc'    : y_obs2_unc/y_obs2,
       'expt_label' : 'exp2',
       'bias_label' : 'b',
       'scale_flag' : False,
       'corr'       : np.eye(len(x_obs2)) }

data3 = {'X'        : x_obs3,
       'expt_value' : y_obs3,
       'rel_unc'    : y_obs3_unc/y_obs3,
       'expt_label' : 'exp3',
       'bias_label' : ['a', 'c'],         # bias label can be a list indicating that this experiment contributes to more than one bias group
       'scale_flag' : False,
       'corr'       : np.eye(len(x_obs3)) }

data4 = {'X'        : x_obs4,
       'expt_value' : y_obs4,
       'rel_unc'    : y_obs4_unc/y_obs4,
       'expt_label' : 'exp4',
       'bias_label' : 'c',
       'scale_flag' : True,
       'corr'       : np.eye(len(x_obs4)) }

In [ ]:
from processing import compile_data_dicts

data_dict = compile_data_dicts([data1, data2, data3, data4], xminmax=[1e-5,12])

# inspect compiled data dict 
print(data_dict.keys())

# Constructing the model and visualizing bias terms

First initialize the BasisModel class which will handle information about the basis functions used to model both the mean/target model and the bias terms.

In [ ]:
from sparse_bias import BiasModel, BasisModel 
from processing  import convert_to_stan_data

############
#
# Build relevant Basis functions
#
basis_obj = BasisModel(data_dict,
                       mean_basis_type = "interpolation",
                       bias_basis_type = "gaussian")

minX = np.log10(1e-3)
maxX = np.log10(25)

basis_obj.generate_mean_bases(X_grid=x_eval) #,centers=centers, widths=width)

# Gaussian bases for the bias - center and width parameters for bias gaussian functions
centers  = [
    np.linspace(minX, maxX, 20), 
    np.linspace(minX, maxX, 15),
    np.linspace(minX, maxX, 10)
]
widths   = [0.1, 0.25, 0.5]
basis_obj.generate_bias_bases(centers=centers, widths=widths) # note in docstring this is logarithm of energy


In [ ]:
### Visualize your bias terms 

plot_x = np.logspace(*x_range, 500)
B_s, B_m, B_l = basis_obj.get_plotable_bias_bases(plot_x)

gamma_s = np.ones((len(centers[0]),1))#*0
gamma_m = np.ones((len(centers[1]),1))
gamma_l = np.ones((len(centers[2]),1))

# gamma_s[7] = 0.5
delta_s = B_s @ gamma_s
delta_m = B_m @ gamma_m*0.5
delta_l = B_l @ gamma_l*0.5

plt.figure(figsize=(5,3))
plt.plot(plot_x, np.exp(delta_s))
plt.plot(plot_x, np.exp(delta_m))
plt.plot(plot_x, np.exp(delta_l))
plt.xlim(1e-3, 12)

plt.xscale('log')



# BiasModel class 
Next, instantiate the BiasModel class by passing an instance of the BasisModel class and a few other options.
The tau_scale is a hyperparameter which controls the level of sparsity imposed by the statistical model. 
The smaller tau is, stronger evidence for a particular bias term is needed for it to deviate from the nominal value.
The magnitude of this value roughly corresponds

The model can be fit using an instance of this class as shown below. 
The number of samples refer to the Monte Carlo samples of the posterior, along with a number of warmup samples.
In general, Markovian statistical samplers should undergo a number of 'warmup' samples before contributing to summary statistic such as mean values or variance. 

Other kwags can be passed to BiasModel.fit() which is be directly passed to the CmdStanModel.sample() function. 

In [ ]:
############
#
# Build Sparse Bias Model
#
sbmod = BiasModel(basis_obj, 
                  model_name = "interpolation_horseshoe",
                  tau_scale=1.e-5) 

# fit
nsamples = 2500
sbmod.fit(n_warmup=2000, n_sample=nsamples, show_console=False) 


In [ ]:
sbmod.output.summary() #kwargs- percentiles=(5,50,95)

# Saving Data

In [ ]:
### all necessary input data and results needed for analysis are held in the BiasModel class which can be saved with pickle

# import pickle
# with open('bias_model.pkl', 'wb') as file:
#     pickle.dump(sbmod, file)

#### and loaded
# with open('bias_model.pkl', 'rb') as file:
#     sbmod = pickle.load(file)

In [ ]:
#### alternatively the stan posterior samples can be saved in a more general csv format - but these won't have all of the information necessary for visualization

# sbmod.output.save_csvfiles("stanout")

# import cmdstanpy
# testoutput = cmdstanpy.from_csv("stanout")

# Analyze Output

To get user started on analysis.

In [ ]:
from utils import analysis
import importlib
importlib.reload(analysis)

myanalysis = analysis.BiasAnalysis(sbmod)
myanalysis.compile_plotable_attributes(plot_x)

In [ ]:

print(f"ilevels: \t\t{[each for each in range(myanalysis.n_levels)]}")

count = myanalysis.count_terms_beyond_threshold_per_level(quantile = 0.7, threshold = 1e-5) # Pq or quantile level (quantile is the actual value)
print(f"Terms above threshold: \t{count}")

integrals = myanalysis.bias_L2norm_per_level()
print(f"Integral Bias: \t\t{np.round(integrals,4)}")

In [ ]:
fig = myanalysis.get_ilevel_figure(0, feature='Feature 0', label_data=False, Nqs=50, )
fig.tight_layout()

In [ ]:
fig = myanalysis.get_ilevel_figure(1, feature='Feature 1', label_data=False, Nqs=50, )
fig.tight_layout()

In [ ]:
### The myanalysis class does the following under the hood

In [ ]:
# # get stan static parameters
# n_bases, datascale_mask, levels_mask, n_levels, B, tau_scale = analysis.unpack_stan_data(sbmod.data_dict)

# # get stan sampled values
# sigma, datascales, [gamma_l, gamma_m, gamma_s] = analysis.read_output(sbmod.output.draws_pd(), n_bases, n_levels, tau_scale)

# # get experimental model 
# exp_model = analysis.get_scaled_model(B, sigma, datascales, datascale_mask)

# # get smooth plotable bias terms 
# B_s, B_m, B_l = basis_obj.get_plotable_bias_bases(plot_x)

# # get dataframe handy for plotting
# all_df = pd.DataFrame({k: sbmod.basis_model.data_dict[k] for k in ['X', 'expt_value', 'rel_unc', 'expt_label', 'bias_label', 'scale_flag']})
# quantile = 0.7
# threshold = 1e-4
# count_per_level = analysis.get_count_beyond_threshold_per_level(n_levels, gamma_s, gamma_m, gamma_l, quantile, threshold)

# print(count_per_level)

In [ ]:

# ilevel = 2

# # for ilevel get experimental model and delta
# delta_exp = analysis.get_delta_for_level(ilevel, sbmod.basis_model.bias_basis_matrix_s, sbmod.basis_model.bias_basis_matrix_m, sbmod.basis_model.bias_basis_matrix_l, gamma_s, gamma_m, gamma_l)
# exp_model_level = analysis.get_corrected_model_for_level(ilevel, levels_mask, exp_model, delta_exp)

# # for ilevel get plotable delta
# delta_plot = analysis.get_delta_for_level(ilevel, B_s, B_m, B_l, gamma_s, gamma_m, gamma_l)

# # filter to level and sort
# x = all_df["X"].values[levels_mask[:,ilevel].astype(bool)]
# exp_model_level = exp_model_level[levels_mask[:,ilevel].astype(bool),:]

# isort = np.argsort(x)
# x = x[isort]
# exp_model_level = exp_model_level[isort,:]

# fig, axes = plt.subplots(2,1, figsize=(8,5), sharex=True, height_ratios=[3,1])

# _ = axes[0].plot(x_eval, np.mean(sigma,axis=0), alpha=1.0, color='k', label="Model", zorder=5)
# _ = axes[0].plot(x, np.mean(exp_model_level, axis=1), 'b', label="Bias Model", alpha=1.0)

# active_exp = all_df[levels_mask[:,ilevel]==1]
# _ = axes[0].errorbar( active_exp["X"],  active_exp["expt_value"], yerr=active_exp["expt_value"]*active_exp["rel_unc"], label=np.unique(active_exp['expt_label']), fmt='.', color="b")

# inactive_exp = all_df[levels_mask[:,ilevel]==0]
# _ = axes[0].errorbar( inactive_exp["X"],  inactive_exp["expt_value"], yerr=inactive_exp["expt_value"]*inactive_exp["rel_unc"], fmt='.', color="k", alpha=0.1)

# # plt.plot(theo_E, np.mean(sigma,axis=0), alpha=1.0, color='b', zorder=5)
# axes[0].set_xlim(10**minX, 10**maxX)
# # axes[0].set_ylim(1e0, 5)
# axes[0].set_xscale('log')
# axes[0].set_yscale('log') 
# # axes[0].set_ylabel("(n,f) cross section")
# axes[0].legend()



# _= axes[1].plot(plot_x, np.exp(delta_plot), 'b', alpha=0.1)
# ydev = 1.1 * np.max(np.abs(1- np.array(axes[1].get_ylim())))
# axes[1].axhline(y=1.0, color='k')
# # axes[1].set_xlim(10**minE, 10**maxE)
# axes[1].set_ylim(1-ydev, 1+ydev)
# axes[1].set_ylabel("Bias")